# Step 3 — Aggregation Pipelines: `$project`, `$lookup`, and more

In this activity we'll work through two guided examples that combine multiple aggregation stages.
By the end you should be comfortable with:

| Stage | What it does |
|---|---|
| `$project` | Reshape documents — rename, compute, include/exclude fields |
| `$unwind` | Flatten an array field into one doc per element |
| `$lookup` | Left-outer-join another collection |
| `$group` | Aggregate values (sum, avg, count, first, etc.) |

These are exactly the tools you'll need for the Step 5 COVID homework.

> **FYI:** There's also a `$out` stage that writes pipeline results to a **new collection** — like `CREATE TABLE AS SELECT` in SQL. We won't use it here since we're on a shared read-only connection, but you'll want it when working on your own Atlas cluster.

---
## Setup

In [ ]:
!pip install pymongo

In [ ]:
import pymongo

user = "class"
password = "184vLpDKvOhvv528"
cluster = "cluster0"
dnsprefix = "wvdjn"
connectionUrl = f"mongodb+srv://{user}:{password}@{cluster}.{dnsprefix}.mongodb.net/"
client = pymongo.MongoClient(connectionUrl)
print(f"Ping result: {client.admin.command('ping')}")

client.admin.command('ping')
print("Connected!")

In [ ]:
db = client["movies"]

# Quick check — you should see movies, ratings, and box_office
print("Collections:", db.list_collection_names())

---
## Example 1 — `$lookup` + `$project` + `$group`: Revenue per year

**Goal:** For each release year, compute the total and average worldwide gross.

This shows how `$project` lets you reshape documents *before* grouping —
here we extract just the fields we care about and rename them for clarity.

In [ ]:
pipeline = [
    # Step 1: Lookup box_office data onto each movie
    {"$lookup": {
        "from": "box_office",
        "localField": "title",
        "foreignField": "title",
        "as": "box"
    }},
    # Step 2: Unwind the joined array (1 box_office doc per movie)
    {"$unwind": "$box"},
    # Step 3: Project only the fields we need, with clean names
    {"$project": {
        "_id": 0,
        "title": 1,
        "year": 1,
        "worldwide_gross": "$box.worldwide_gross"
    }},
    # Step 4: Group by year
    {"$group": {
        "_id": "$year",
        "total_gross": {"$sum": "$worldwide_gross"},
        "avg_gross":   {"$avg": "$worldwide_gross"},
        "movie_count": {"$sum": 1}
    }},
    {"$sort": {"_id": 1}}
]

for doc in db.movies.aggregate(pipeline):
    print(doc)

### Key takeaways
- `$project` acts like SQL `SELECT col AS alias` — it reshapes every document.
- You can reference nested/joined fields with dot notation like `"$box.worldwide_gross"`.
- Stages execute **in order** — a pipeline is just a list of transformations.

---
## Example 2 — `$lookup` + `$unwind` + `$project`: Flatten movies with ratings

**Goal:** Produce a clean, flat view of each movie with its `title`, `genre`, `country`, and `rating`.

This is a common pattern: join → unwind → project. It's the MongoDB equivalent of a SQL `SELECT ... JOIN ... ON ...`.

In [ ]:
pipeline = [
    # Join ratings onto movies
    {"$lookup": {
        "from": "ratings",
        "localField": "title",
        "foreignField": "title",
        "as": "rating_info"
    }},
    # preserveNullAndEmptyArrays = True keeps movies even if no rating (LEFT JOIN)
    {"$unwind": {"path": "$rating_info", "preserveNullAndEmptyArrays": True}},
    # Clean output — only the fields we want, with a fallback for missing ratings
    {"$project": {
        "_id": 0,
        "title": 1,
        "genre": 1,
        "country": 1,
        "rating": {"$ifNull": ["$rating_info.rating", "N/A"]}
    }}
]

for doc in db.movies.aggregate(pipeline):
    print(doc)

### Key takeaways
- `preserveNullAndEmptyArrays: True` makes `$unwind` behave like a **LEFT JOIN** instead of an INNER JOIN.
- `$ifNull` provides a default when a field is missing — handy for keeping output clean.
- This "join → unwind → project" pattern is one you'll use constantly.

---
## Now you try!

Use the same `movies`, `ratings`, and `box_office` collections for the exercises below.

### Exercise 1

Find the **top-rated movie per genre**.

Hints:
- `$lookup` ratings onto movies
- `$unwind` the ratings
- `$project` to keep `title`, `genre`, and `rating`
- `$sort` by rating descending, then `$group` by genre with `$first`

In [ ]:
# your code here

### Exercise 2

For each genre, compute:
- `movie_count` — number of movies
- `avg_worldwide_gross` — average box office worldwide gross
- `avg_rating` — average rating

Sort by `avg_worldwide_gross` descending.

Hints:
- You'll need **two** `$lookup` stages (one for ratings, one for box_office)
- `$unwind` each joined array
- `$group` by genre with `$avg` and `$sum`
- Use `$project` at the end to rename `_id` to `genre` and round the averages

In [ ]:
# your code here